# LLamaFactory

使用llamafactory来进行0代码微调，需要配置的文件有两个:
1. dataset_info.json 文件 主要是用来注册数据集，注册好的数据直接放到lora_sft.yaml中使用
2. lora_sft.yaml 用来配置llamafactory里面的各种数据



In [1]:
!type dataset_info.json

{
  "anti_fraud_0819_train": {
    "file_name": "../datasets/11/train_test/train0819_alpaca.jsonl",
    "columns": {
      "prompt": "instruction",
      "query": "input",
      "response": "output"
    }
  },
  "anti_fraud_0819_eval": {
    "file_name": "../datasets/11/train_test/eval0819_alpaca.jsonl",
    "columns": {
      "prompt": "instruction",
      "query": "input",
      "response": "output"
    }
  },
  "anti_fraud_0819_test": {
    "file_name": "../datasets/11/train_test/test0819_alpaca.jsonl",
    "columns": {
      "prompt": "instruction",
      "query": "input",
      "response": "output"
    }
  }
}


anti_fraud_0819_train 和 anti_fraud_0819_eval 和 anti_fraud_0819_test 是注册好的数据集名字，放在lora_sft.yaml中使用

In [3]:
!type anti_fraud_qwen2_lora_sft.yaml

dataset_dir: E:/Code/StudyLLM/11.Anti-Frad
model_name_or_path: E:/Code/StudyLLM/models/Qwen2-0.5B-Instruct
trust_remote_code: true

stage: sft
do_train: true
finetuning_type: lora

dataset: anti_fraud_0819_train
eval_dataset: anti_fraud_0819_eval
template: qwen
cutoff_len: 2048
overwrite_cache: true
preprocessing_num_workers: 4

output_dir: E:/Code/StudyLLM/11.Anti-Frad/models/Qwen2-0.5B-Instruct_fr_0819_lf
overwrite_output_dir: true
logging_steps: 10
save_steps: 10
eval_steps: 10
eval_strategy: steps
load_best_model_at_end: true

per_device_train_batch_size: 4
per_device_eval_batch_size: 4
gradient_accumulation_steps: 4
learning_rate: 1.0e-4
num_train_epochs: 3
bf16: true
gradient_checkpointing: true

lora_rank: 8
lora_alpha: 16
lora_dropout: 0.05
lora_target: q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj

plot_loss: true


In [ ]:
# AutoDL/Linux: 如果上传后当前目录不在第 11 章目录，先切换到这里
# 按你的实际上传路径修改这一行。例如：/root/autodl-tmp/StudyLLM/11.Anti-Frad
%cd /root/autodl-tmp/StudyLLM/11.Anti-Frad

## AutoDL 1：检查数据和配置文件

在 AutoDL 上是 Linux 环境，所以查看文件用 `cat`，不是 Windows 的 `type`。

In [ ]:
!pwd
!ls -lh
!ls -lh data/train_test
!cat dataset_info.json
!cat anti_fraud_qwen2_lora_sft.yaml

## AutoDL 2：下载并安装 LLaMA-Factory

这一步只需要在 AutoDL 的训练环境里执行一次。后面如果已经安装过，可以跳过。

In [ ]:
%%bash
set -e
cd /root/autodl-tmp
if [ ! -d LLaMA-Factory ]; then
  git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
fi
cd LLaMA-Factory
pip install -e .
pip install -r requirements/metrics.txt
llamafactory-cli version || true

## AutoDL 3：确认环境能调用 CLI 和 GPU

In [ ]:
!which llamafactory-cli
!nvidia-smi
!python -c "import torch; print('torch:', torch.__version__); print('cuda:', torch.version.cuda); print('available:', torch.cuda.is_available())"

## AutoDL 4：先小样本测试训练

先用 `max_samples=100` 跑通数据读取、模型加载、LoRA 注入和 checkpoint 保存。能跑通后再全量训练。

这里只训练前100个样本

In [ ]:
!llamafactory-cli train anti_fraud_qwen2_lora_sft.yaml max_samples=100

## AutoDL 5：全量训练

小样本测试通过后再运行这一格。

In [ ]:
!llamafactory-cli train anti_fraud_qwen2_lora_sft.yaml